In [2]:
import requests
import pandas as pd
import re
from bs4 import BeautifulSoup
from io import StringIO

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

def get_meta(soup, name):
    """Pull a <meta name="..." content="..."> value, or None if missing."""
    tag = soup.find("meta", attrs={"name": name})
    return tag["content"].strip() if tag and tag.get("content") else None

def scrape_result_page(url):
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    # --- metadata sitting in the page's <meta> tags ---
    meet_name  = get_meta(soup, "EventName")
    event_date = get_meta(soup, "EventDate")
    round_name = get_meta(soup, "PhaseName")
    sex        = get_meta(soup, "Sex")

    # --- wind reading lives in a plain heading like "20 AUG 2023 19:10 Wind 0.0" ---
    wind = None
    heading = soup.find(string=lambda s: s and "Wind" in s)
    if heading:
        match = re.search(r"Wind\s*(-?\d+\.\d+)", heading)
        if match:
            wind = float(match.group(1))

    # --- the actual results table ---
    tables = pd.read_html(StringIO(resp.text))
    if not tables:
        raise ValueError(f"No table found at {url}")
    df = max(tables, key=len)   # results table is the largest table on the page

    # attach metadata to every row so nothing gets lost once we combine meets
    df["meet_name"]  = meet_name
    df["event_date"] = event_date
    df["round"]      = round_name
    df["sex"]        = sex
    df["wind"]       = wind
    df["source_url"] = url

    return df

# test on the 2023 Worlds 100m final
url = "https://worldathletics.org/results/world-athletics-championships/2023/world-athletics-championships-budapest-2023-7138987/men/100-metres/final/result"
df = scrape_result_page(url)
df

,POS,BIB,ATHLETE,COUNTRY,MARK,ms*,Reaction Time,meet_name,event_date,round,sex,wind,source_url
0,1,3007,Noah LYLES,USA,9.83 =WL,NaN,0.145,"World Athletics Championships, Budapest 2023",19/08/2023 00:00:00,100 Metres,men,0.0,https://worldathletics.org/results/world-athle...
1,2,2102,Letsile TEBOGO,BOT,9.88 NR,0.873,0.129,"World Athletics Championships, Budapest 2023",19/08/2023 00:00:00,100 Metres,men,0.0,https://worldathletics.org/results/world-athle...
2,3,2350,Zharnel HUGHES,GBR,9.88,0.874,0.105,"World Athletics Championships, Budapest 2023",19/08/2023 00:00:00,100 Metres,men,0.0,https://worldathletics.org/results/world-athle...
3,4,2554,Oblique SEVILLE,JAM,9.88,0.877,0.148,"World Athletics Championships, Budapest 2023",19/08/2023 00:00:00,100 Metres,men,0.0,https://worldathletics.org/results/world-athle...
4,5,2978,Christian COLEMAN,USA,9.92,NaN,0.137,"World Athletics Championships, Budapest 2023",19/08/2023 00:00:00,100 Metres,men,0.0,https://worldathletics.org/results/world-athle...
5,6,2587,Abdul Hakim SANI BROWN,JPN,10.04,NaN,0.177,"World Athletics Championships, Budapest 2023",19/08/2023 00:00:00,100 Metres,men,0.0,https://worldathletics.org/results/world-athle...
6,7,2626,Ferdinand OMANYALA,KEN,10.07,NaN,0.152,"World Athletics Championships, Budapest 2023",19/08/2023 00:00:00,100 Metres,men,0.0,https://worldathletics.org/results/world-athle...
7,8,2540,Ryiem FORDE,JAM,10.08,NaN,0.188,"World Athletics Championships, Budapest 2023",19/08/2023 00:00:00,100 Metres,men,0.0,https://worldathletics.org/results/world-athle...
